### Data Connect Hub — SDK quickstart

Runs the SDK against a live Data Connect Hub deployment.

**Prerequisites**

1. A Data Connect Hub cluster is up and running with the gateway accessible from your machine.
2. A **tenant namespace** with a data connection and a **service account**.
3. A **connector secret** with credentials for your data source already exists in that tenant namespace.
4. The Python SDK installed with the Flight extra.

**Configuration**

Update the environment variables below to match the gateway host and credentials provided by your cluster administrator:

In [97]:
import os

DCH_HOST = os.getenv("DCH_HOST", "https://rh-ai.apps.rosa.c5t45-96i4k-aam.l9yc.p3.openshiftapps.com")  # gateway host or host:port
TENANT_NAMESPACE = os.getenv("DCH_TENANT_NAMESPACE", "dch-tenant-a")  # replace with the namespace where the data connection resides
DB_SECRET_NAME = os.getenv("DCH_DB_SECRET_NAME", "my-db-creds")
INSECURE = os.getenv("DCH_INSECURE", "true").lower() in ("1", "true", "yes")
CA_CERT = os.getenv("DCH_CA_CERT") or None

# Every REST and Flight call sends x-tenant-id; the services reject an empty value.
if not TENANT_NAMESPACE:
    raise ValueError("Set DCH_TENANT_NAMESPACE to the namespace where the data connection resides")

In [98]:
from data_connect_hub import DataConnectClient

# Option 1: paste a token string obtained from your cluster administrator
TOKEN = os.getenv("DCH_TOKEN", "eyJhbGciOiJSUzI1NiIsImtpZCI6InA2NmxtWG5xbEtIaGMycW4xS2YteHlQY18zOG9CNUhPd1RyTjl3eGpCSjQifQ.eyJhdWQiOlsiaHR0cHM6Ly9yaC1vaWRjLnMzLnVzLWVhc3QtMS5hbWF6b25hd3MuY29tLzI3YmQ2Y2cwdnM3bm4wOG11ZTgzZmJvZjk0ZGo0bTlhIl0sImV4cCI6MTc5NTEzMjk2OSwiaWF0IjoxNzk1MTI5MzY5LCJpc3MiOiJodHRwczovL3JoLW9pZGMuczMudXMtZWFzdC0xLmFtYXpvbmF3cy5jb20vMjdiZDZjZzB2czdubjA4bXVlODNmYm9mOTRkajRtOWEiLCJqdGkiOiJjYmQyZDgzNy02ZTY5LTQxZGEtOTdhMi02ZmI4OWUwMjYzMGUiLCJrdWJlcm5ldGVzLmlvIjp7Im5hbWVzcGFjZSI6ImRjaC10ZW5hbnQtYSIsInNlcnZpY2VhY2NvdW50Ijp7Im5hbWUiOiJkY2gtdGVzdC11c2VyIiwidWlkIjoiMDNhZTY4NGItMGRjOS00YzAxLWFmZTYtNjA5ZTc4NjQ5NGMxIn19LCJuYmYiOjE3OTUxMjkzNjksInN1YiI6InN5c3RlbTpzZXJ2aWNlYWNjb3VudDpkY2gtdGVuYW50LWE6ZGNoLXRlc3QtdXNlciJ9.pPmYHPs9zxuGawC0Q9PemDNgrZ3YWNiThZm9VLnzmfLxART9tZo2He-3-bRzDrhMtfny6GQVlKUAX2LCKEdDEMBC0qJ6w0yJfsNzirLq9bN6Q7oQuON9wRjn6tRYAJPgOZBQrF7O-B82Iu__H16VvVKfa6nIy_kfYAfqGhDaTYa25Fr_X8vNp5Yf_MmwEfqm8Yll00bJpO5EBDuQBJmwqhQT6c1cOnSWzH1mJqbK7UWgPekvj-zFgbb3JJ693xfSKZI4QDDLGUP8vpkYjcM3QBYtb-SFzvuOJHt8OZryvSfAKyzlLU0-3I2Wd8voT1vQkNZDOf5drezSXms4aYtWFU8slhlLesrVFZF1qLouvwyvPepeIQ3I-e6uBTIGL7iXfIKdAFpq0XDexTbEPqaYPeYdHAGmzhoT-4deJaRzVp6RIvll5Fdh2d8RnwtYJCZqNk1xdx3blPdT_DgPCz8m98Bopv3AXV_4RPvlTEDVwRPfz4y6cfcxmPUX5dwnUOsDHDL9hK9UZfrQz-63Ua67toStFelBehK-r0CIgDWbQiRzq468UDHnHXSBRrOcoS1NoNKzAhMxR9rneVQTt8FUku0E5e6RtjsQ8gCcz2F90LAg8f_lZ3hfCVe6UGuk_J8Mfure7rnhnv27gVi_iQl1gWNwJj1WMvQ7-jZYv-p-0jg")

# Option 2: if you have the oc CLI, request a short-lived token automatically.
# The SDK calls the provider once, caches the token, and refreshes it on a 401.
# SA_NAME = os.getenv("DCH_SA_NAME", "dch-user")
# SA_ISSUER = os.getenv("DCH_SA_ISSUER", "https://kubernetes.default.svc")
# import subprocess
# def sa_token() -> str:
#     return subprocess.check_output(
#         ["oc", "create", "token", SA_NAME, "-n", TENANT_NAMESPACE,
#          "--audience", SA_ISSUER, "--duration", "3600s"],
#         text=True,
#     ).strip()

client = DataConnectClient(
    DCH_HOST,
    token=TOKEN,  # for Option 2, replace with: token_provider=sa_token,
    tenant_id=TENANT_NAMESPACE,
    ca_cert=CA_CERT,
    insecure=INSECURE,
)
print(f"Client ready — gateway: {DCH_HOST}")

Client ready — gateway: https://rh-ai.apps.rosa.c5t45-96i4k-aam.l9yc.p3.openshiftapps.com


**Connection types (REST)**

In [99]:
types = client.list_connection_types()
print(f"Found {len(types)} connection type(s):")
for ct in types:
    print(f"  [{ct.id}] {ct.name}: {ct.description}")

Found 8 connection type(s):
  [54533b68-cca9-409d-b716-6bd6d26ba752] ElasticSearch: Elasticsearch search and analytics engine
  [c07df15b-b921-4537-ab5c-93ab1ad6f86f] Neo4j: Neo4j graph database
  [1a02e827-79c3-4008-b349-6e19441d4139] PGVector: PostgreSQL vector database connection type
  [6c0f97d5-d06e-4cca-8d1e-59ea09cebb5e] HuggingFace: Hugging Face model and dataset hub
  [97d0d6eb-286b-404a-8cb2-e619259f589b] URI: HTTP/REST JSON API endpoint
  [0eab5eb1-867c-4d1a-80e1-081643f87506] Postgres: Postgres data connection type
  [1b26499e-3f6f-44d8-b6ac-a1394f6bfacf] Milvus: Milvus vector database
  [4be3a4cb-8513-4300-a276-6d311e466771] test-postgres-1: test connection type


In [ ]:
new_type = client.create_connection_type(
    name="example-postgres",
    provider="postgres",
    description="PostgreSQL connector created by quickstart",
)
print(f"Created type: {new_type.id} ({new_type.name})")

In [ ]:
fetched = client.get_connection_type(new_type.id)
print(f"Fetched: {fetched.name} — {fetched.description}")

In [ ]:
updated_type = client.update_connection_type(
    new_type.id,
    description="PostgreSQL connector — updated description",
)
print(f"Updated type {updated_type.id}: {updated_type.description}")

**Connections (REST)**

Connections reference a connection type. The cell above must have run first.

In [102]:
connections = client.list_connections()
print(f"Found {len(connections)} connection(s):")
for c in connections:
    print(f"  [{c.id}] {c.name} (type={c.data_connection_type_id}, format={c.format})")

Found 0 connection(s):


In [101]:
client.delete_connection("2a23a382-5e93-47f7-9fb3-0fb94810ce68")

In [ ]:
from data_connect_hub import CredentialsRef

new_conn = client.create_connection(
    name="quickstart-db",
    connection_type_id=new_type.id,
    data_format="tabular",
    credentials_ref=CredentialsRef(secret=DB_SECRET_NAME),
)
print(f"Created connection: {new_conn.id} ({new_conn.name})")

In [ ]:
fetched_conn = client.get_connection(new_conn.id)
print(f"Fetched: {fetched_conn.name} (type={fetched_conn.data_connection_type_id}, format={fetched_conn.format})")

In [ ]:
updated_conn = client.update_connection(
    new_conn.id,
    name="quickstart-db-updated",
)
print(f"Updated connection {updated_conn.id}: {updated_conn.name}")

**Flight SQL**

Query data through the Flight SQL interface. Requires a connection whose database is reachable from the Flight service.

In [ ]:
CONNECTION_ID = new_conn.id  # or paste an existing connection ID

In [ ]:
info = client.server_info()
print("Server info:")
for key, value in info.items():
    print(f"  {key}: {value}")

In [ ]:
# Retrieve table metadata from the database
tables = client.get_tables(CONNECTION_ID)
print(f"Available tables: {tables.num_rows} table(s)")
if tables.num_rows > 0:
    print(tables.to_pandas()[['catalog_name', 'db_schema_name', 'table_name', 'table_type']])

In [ ]:
QUERY = "SELECT 1 AS hello"  # replace with a query for your database

table = client.read(QUERY, CONNECTION_ID)
print(f"PyArrow Table — {table.num_rows} row(s), {table.num_columns} column(s)")
print(table.to_pydict())

In [ ]:
for i, batch in enumerate(client.read_batches(QUERY, CONNECTION_ID)):
    print(f"Batch {i}: {batch.num_rows} row(s)")
print("Stream exhausted — resources released.")


In [ ]:
df = client.read_pandas(QUERY, CONNECTION_ID)
df

**Cleanup**

Delete the resources created by this notebook.

In [ ]:
client.delete_connection(new_conn.id)
print(f"Deleted connection: {new_conn.id}")

In [ ]:
client.delete_connection_type(new_type.id)
print(f"Deleted connection type: {new_type.id}")